In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import numpy as np
import timeit
import csv
import numba
from numba import set_num_threads
from sklearn.datasets import make_regression

import sys

In [2]:
current_dir = os.getcwd()
# current_dir

In [3]:
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(current_dir), '../')))

In [4]:
# os.path.dirname(current_dir)

In [5]:
# sys.path

In [6]:
# -----------------------------
# IMPORT YOUR FUNCTIONS
# -----------------------------
from Experiments.Numba.OMP_Implementations.OMP_numba_parallel import (
    OMP_serial,
    OMP_numba_serial,
    OMP_numba_jit,
    OMP_numba_njit,
    OMP_numba_njit_parallel,
    OMP_numba_njit_parallel_pbatches,
)

In [7]:
import importlib
OMP_A_V1_1 = importlib.import_module('Implementations.AK-SVD.Approx_V1_1').OMP
OMP_NA_V0 = importlib.import_module('Implementations.AK-SVD.Numba_Approx_V0').OMP
OMP_NA_V1 = importlib.import_module('Implementations.AK-SVD.Numba_Approx_V1').OMP

In [8]:
# -----------------------------
# DATA SETUP
# -----------------------------
D, y = make_regression(
    n_samples=50,
    n_features=300,
    n_targets=20000,
    noise=4,
    random_state=0
)

Y = y.T.astype(np.float32)
D = D.T.astype(np.float32)

n_samples = Y.shape[0]

In [9]:
# -----------------------------
# PARAMS
# -----------------------------
T_0 = 1
N = 10  # timeit runs

functions = [
    ("OMP_serial", OMP_serial),
    # ("OMP_numba_serial", OMP_numba_serial),
    ("OMP_numba_jit", OMP_numba_jit),
    ("OMP_numba_njit", OMP_numba_njit),
    # ("OMP_numba_njit_parallel", OMP_numba_njit_parallel),
    ("OMP_numba_njit_parallel_pbatches", OMP_numba_njit_parallel_pbatches),
    
    ("Approx_V1_1", OMP_A_V1_1),
    ("Numba_Approx_V0", OMP_NA_V0),
    ("Numba_Approx_V1", OMP_NA_V1),   
]

max_threads = numba.get_num_threads()

thread_list = [1]
batch_sizes = [2**i for i in range(1, int(np.log2(n_samples/16)) + 1)]
batch_sizes = [b for b in batch_sizes if b <= n_samples]

print("Threads:", thread_list)
print("Batch sizes:", batch_sizes)

Threads: [1]
Batch sizes: [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]


In [10]:
# -----------------------------
# WARMUP (compile all functions)
# -----------------------------
for name, fn in functions:
    print(name)
    try:
        fn(Y, T_0, D, batch_size=2, float_dtype=np.float32)
    except Exception as e:
        print(e)
        try:
            fn(Y, T_0, D, batch_size=2)
        except Exception as f:
            print(f)
        print()

OMP_serial
OMP_serial() got an unexpected keyword argument 'float_dtype'

OMP_numba_jit
some keyword arguments unexpected


C:\Users\richa\Jupyter Notebooks\CSE 392 - Parallel Algorithms\kSVD\Experiments\Numba\OMP_Implementations\OMP_numba_parallel.py:349: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float32, 2, 'A', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  dot = Dk_new[b] @ Q[b, :, :j]


Failed in nopython mode pipeline (step: nopython frontend)
Failed in nopython mode pipeline (step: nopython frontend)
No implementation of function Function(<intrinsic _impl>) found for signature:
 
 >>> _impl(array(float32, 2d, A), array(float64, 2d, A))
 
There are 2 candidate implementations:
 - Of which 2 did not match due to:
 Intrinsic in function 'dot_2_impl.<locals>._impl': File: numba\np\linalg.py: Line 554.
   With argument(s): '(array(float32, 2d, A), array(float64, 2d, A))':
  Rejected as the implementation raised a specific error:
    TypingError: '@' arguments must all have the same dtype
  raised from C:\Users\richa\miniforge3\envs\ml_stable\Lib\site-packages\numba\np\linalg.py:574

During: resolving callee type: Function(<intrinsic _impl>)
During: typing of call at C:\Users\richa\miniforge3\envs\ml_stable\Lib\site-packages\numba\np\linalg.py (593)

File "..\..\..\..\..\miniforge3\envs\ml_stable\Lib\site-packages\numba\np\linalg.py", line 593:
            def _dot2_codeg

C:\Users\richa\Jupyter Notebooks\CSE 392 - Parallel Algorithms\kSVD\Experiments\Numba\OMP_Implementations\OMP_numba_parallel.py:579: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float32, 2, 'A', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  dot = Dk_new[b] @ Q[b, :, :j]


Failed in nopython mode pipeline (step: nopython frontend)
Failed in nopython mode pipeline (step: nopython frontend)
No implementation of function Function(<intrinsic _impl>) found for signature:
 
 >>> _impl(array(float32, 2d, A), array(float64, 2d, A))
 
There are 2 candidate implementations:
 - Of which 2 did not match due to:
 Intrinsic in function 'dot_2_impl.<locals>._impl': File: numba\np\linalg.py: Line 554.
   With argument(s): '(array(float32, 2d, A), array(float64, 2d, A))':
  Rejected as the implementation raised a specific error:
    TypingError: '@' arguments must all have the same dtype
  raised from C:\Users\richa\miniforge3\envs\ml_stable\Lib\site-packages\numba\np\linalg.py:574

During: resolving callee type: Function(<intrinsic _impl>)
During: typing of call at C:\Users\richa\miniforge3\envs\ml_stable\Lib\site-packages\numba\np\linalg.py (593)

File "..\..\..\..\..\miniforge3\envs\ml_stable\Lib\site-packages\numba\np\linalg.py", line 593:
            def _dot2_codeg

C:\Users\richa\Jupyter Notebooks\CSE 392 - Parallel Algorithms\kSVD\Experiments\Numba\OMP_Implementations\OMP_numba_parallel.py:1039: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float32, 2, 'A', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  dot = Dk_new[b] @ Q[b, :, :j]


Failed in nopython mode pipeline (step: nopython frontend)
Failed in nopython mode pipeline (step: nopython frontend)
No implementation of function Function(<intrinsic _impl>) found for signature:
 
 >>> _impl(array(float32, 2d, A), array(float64, 2d, A))
 
There are 2 candidate implementations:
 - Of which 2 did not match due to:
 Intrinsic in function 'dot_2_impl.<locals>._impl': File: numba\np\linalg.py: Line 554.
   With argument(s): '(array(float32, 2d, A), array(float64, 2d, A))':
  Rejected as the implementation raised a specific error:
    TypingError: '@' arguments must all have the same dtype
  raised from C:\Users\richa\miniforge3\envs\ml_stable\Lib\site-packages\numba\np\linalg.py:574

During: resolving callee type: Function(<intrinsic _impl>)
During: typing of call at C:\Users\richa\miniforge3\envs\ml_stable\Lib\site-packages\numba\np\linalg.py (593)

File "..\..\..\..\..\miniforge3\envs\ml_stable\Lib\site-packages\numba\np\linalg.py", line 593:
            def _dot2_codeg

In [11]:
# -----------------------------
# BENCHMARK
# -----------------------------
results = []

for name, fn in functions:
    print(f"\n--- {name} ---")
    
    for n_threads in thread_list:
        set_num_threads(n_threads)
        
        for batch_size in batch_sizes:
            # define callable for timeit
            stmt = lambda: fn(Y, T_0, D, batch_size=batch_size)
            stmt2 = lambda: fn(Y, T_0, D, batch_size=batch_size, float_dtype=np.float32)
            
            
            try:
                stmt2()
            except:
                try:
                    stmt()
                except:
                    print(f"FAIL: {name}, threads={n_threads}, batch={batch_size}")
                    continue
            try:
                t = timeit.timeit(stmt2, number=N) / N
            except Exception as e:
                try:
                    t = timeit.timeit(stmt, number=N) / N               
                except:
                    print(f"FAIL: {name}, threads={n_threads}, batch={batch_size}")
                    continue

            print(f"{name} | threads={n_threads} | batch={batch_size} | {t:.6f}s")

            results.append((name, n_threads, batch_size, t))


--- OMP_serial ---
OMP_serial | threads=1 | batch=2 | 0.862597s
OMP_serial | threads=1 | batch=4 | 0.460553s
OMP_serial | threads=1 | batch=8 | 0.253854s
OMP_serial | threads=1 | batch=16 | 0.127772s
OMP_serial | threads=1 | batch=32 | 0.091648s
OMP_serial | threads=1 | batch=64 | 0.058945s
OMP_serial | threads=1 | batch=128 | 0.039013s
OMP_serial | threads=1 | batch=256 | 0.037953s
OMP_serial | threads=1 | batch=512 | 0.039357s
OMP_serial | threads=1 | batch=1024 | 0.045323s

--- OMP_numba_jit ---
FAIL: OMP_numba_jit, threads=1, batch=2
FAIL: OMP_numba_jit, threads=1, batch=4
FAIL: OMP_numba_jit, threads=1, batch=8
FAIL: OMP_numba_jit, threads=1, batch=16
FAIL: OMP_numba_jit, threads=1, batch=32
FAIL: OMP_numba_jit, threads=1, batch=64
FAIL: OMP_numba_jit, threads=1, batch=128
FAIL: OMP_numba_jit, threads=1, batch=256
FAIL: OMP_numba_jit, threads=1, batch=512
FAIL: OMP_numba_jit, threads=1, batch=1024

--- OMP_numba_njit ---
FAIL: OMP_numba_njit, threads=1, batch=2
FAIL: OMP_numba_nj

In [12]:
# -----------------------------
# SAVE CSV
# -----------------------------
with open("omp_benchmark_all_thread_limit_FP32.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["function", "threads", "batch_size", "time_sec", 'stdev'])
    writer.writerows(results)

print("\nSaved to omp_benchmark_all_thread_limit_FP32.csv")


Saved to omp_benchmark_all_thread_limit_FP32.csv
